# Enterprise BFSI Client MDM Platform

## 01 — Bronze Ingestion

#### Purpose

Persist ADF-landed source files from ADLS Gen2 into Delta Bronze
tables registered in Unity Catalog.

#### Input

ADLS Gen2 `raw/`:

- Core
- CRM
- KYC
- Wealth

#### Processing

1. Read source CSV using explicit source schema.
2. Preserve source values without business standardization.
3. Add `batch_id`.
4. Add `ingestion_timestamp`.
5. Add `record_hash`.
6. Write each source to a Unity Catalog Delta table.

#### Output

- `bfsi_mdm_dev.bronze.core`
- `bfsi_mdm_dev.bronze.crm`
- `bfsi_mdm_dev.bronze.kyc`
- `bfsi_mdm_dev.bronze.wealth`

#### Design principle

Bronze preserves source representation.
Contract validation and standardization occur downstream.

### Purpose:
####   Read the ADF-landed source files from ADLS raw/
####   and persist them as Bronze Delta tables.

Bronze intentionally preserves source values.

Contract validation and standardization happen later.

In [0]:
import uuid

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    StructField,
    StructType,
)

#### Explicit Bronze schemas


We intentionally read source values as STRING.

Bronze is a faithful source-preservation layer.

Contract validation later decides whether a value is valid.

In [0]:
CORE_SCHEMA = StructType([
    StructField("customer_id", StringType(), True),
    StructField("full_name",   StringType(), True),
    StructField("dob",         StringType(), True),
    StructField("phone",       StringType(), True),
    StructField("email",       StringType(), True),
    StructField("address",     StringType(), True),
    StructField("account_id",  StringType(), True),
    StructField("branch_id",   StringType(), True),
])

CRM_SCHEMA = StructType([
    StructField("crm_customer_id",     StringType(), True),
    StructField("customer_name",       StringType(), True),
    StructField("date_of_birth",       StringType(), True),
    StructField("mobile",              StringType(), True),
    StructField("email_address",       StringType(), True),
    StructField("residential_address", StringType(), True),
    StructField("segment",             StringType(), True),
])

KYC_SCHEMA = StructType([
    StructField("kyc_id",            StringType(), True),
    StructField("legal_name",        StringType(), True),
    StructField("dob",               StringType(), True),
    StructField("tax_id",            StringType(), True),
    StructField("phone",             StringType(), True),
    StructField("address",           StringType(), True),
    StructField("kyc_status",        StringType(), True),
    StructField("verification_date", StringType(), True),
    StructField("LEI",               StringType(), True),
])

WEALTH_SCHEMA = StructType([
    StructField("client_id",            StringType(), True),
    StructField("client_name",          StringType(), True),
    StructField("dob",                  StringType(), True),
    StructField("phone",                StringType(), True),
    StructField("email",                StringType(), True),
    StructField("address",              StringType(), True),
    StructField("aum",                  StringType(), True),
    StructField("risk_profile",         StringType(), True),
    StructField("relationship_manager", StringType(), True),
])

#### ADLS RAW INPUT CONFIGURATION

ADF has already copied the four files into these locations.

Databricks now reads those files.

In [0]:
RAW_ROOT = "abfss://raw@stbfsimdmdev.dfs.core.windows.net/"

SOURCES = {
    "core": (
        "core/core_customers.csv",
        CORE_SCHEMA,
    ),
    "crm": (
        "crm/crm_customers.csv",
        CRM_SCHEMA,
    ),
    "kyc": (
        "kyc/kyc_customers.csv",
        KYC_SCHEMA,
    ),
    "wealth": (
        "wealth/wealth_customers.csv",
        WEALTH_SCHEMA,
    ),
}

CATALOG = "bfsi_mdm_dev"

####Generic Bronze reader

The same function handles Core, CRM, KYC and Wealth.

In [0]:
def read_bronze(
    relative_path: str,
    schema: StructType,
    batch_id: str,
):
    """
    Read one source CSV into a Bronze DataFrame.

    Parameters
    ----------
    relative_path:
        Location of the file below the ADLS raw container.

    schema:
        Explicit source schema.

    batch_id:
        Unique ID for this execution of the Bronze pipeline.
    """
    
    df = (
        spark.read
        .option("header", True)
        .option("multiline", True)
        .schema(schema)
        .csv(RAW_ROOT + relative_path)
    )
    
    

    # --------------------------------------------------------
    # Create a deterministic fingerprint of the source row.
    #
    # We concatenate all original source columns and hash
    # them with SHA-256.
    #
    # record_hash is useful later for identifying whether the
    # source content actually changed.
    # --------------------------------------------------------

    concat_all = F.concat_ws(
        "|",
        *[
            F.coalesce(F.col(c), F.lit(""))
            for c in df.columns
        ],
    )

    return (
        df
        # One execution ID shared by all records in this run.
        .withColumn(
            "batch_id",
            F.lit(batch_id),
        )

        # Timestamp when Databricks processed the row.
        .withColumn(
            "ingestion_timestamp",
            F.current_timestamp(),
        )

        # SHA-256 fingerprint of the source content.
        .withColumn(
            "record_hash",
            F.sha2(concat_all, 256),
        )
    )

####One unique ID for this entire Bronze execution.

All four source tables written by this notebook will share this batch_id.

In [0]:
batch_id = str(uuid.uuid4())

print(f"Bronze batch_id: {batch_id}")

Bronze batch_id: 316d99a2-de25-4de9-b9de-a7bc440c6eba


####BRONZE SMOKE TEST ( CORE ONLY )

#### Purpose:
   Verify:
     1. Databricks can read ADLS raw.
     2. Spark can apply the explicit schema.
     3. Metadata columns are created.
     4. Delta writing works.
     5. Unity Catalog registration works.

In [0]:
system = "core"

path, schema = SOURCES[system]

core_bronze_df = read_bronze(
    relative_path=path,
    schema=schema,
    batch_id=batch_id,
)

print("Core Bronze schema:")
core_bronze_df.printSchema()

print("Core Bronze row count:")
print(f"{core_bronze_df.count():,}")

Core Bronze schema:
root
 |-- customer_id: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- email: string (nullable = true)
 |-- address: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- branch_id: string (nullable = true)
 |-- batch_id: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- record_hash: string (nullable = true)

Core Bronze row count:
964,240


In [0]:
display(
    core_bronze_df.limit(10)
)

customer_id,full_name,dob,phone,email,address,account_id,branch_id,batch_id,ingestion_timestamp,record_hash
C00000001,Connor Smith,1971-10-19,+44(0)20 7496 0962,elaine13@example.org,"Studio 8 Hopkins park, West Scott, South East, RG16 2RP",ACC000000001,BR001,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:04:06.626053Z,8102a00df6062ec45f69cc250d4a9dc206186a10f2f9f93dbdf8faa0c30885b6
C00000002,Kevin Ruth Hayes,1970-05-26,01632 960381,jasminehopkins@example.net,"881 Davies hollow, Charlesborough, South East, SO6 2NK",ACC000000002,BR002,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:04:06.626053Z,153bf97e864a23a140285f0ca4b002c28e0b95cf524a46206c55abe328a39ad6
C00000003,Jemma Dean,1957-03-03,+44117 4960356,lewisbrian@example.net,"7 Donald trace, Wallacefurt, Scotland, PA5 2ZM",ACC000000003,BR003,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:04:06.626053Z,1f086542eeb84706310d64d94eaace4972020089c9e2fa7d2f0374363aa6273f
C00000004,Richard Duncan Chambers,1983-10-21,(0909) 879 0389,walkergraham@example.org,"187 Fowler via, North Shaneborough, South East, TN17 7LC",ACC000000004,BR004,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:04:06.626053Z,45daba61a36460eae163d3f55785c3c1185ab30008440d965f7bbe90b8c8e8f5
C00000005,Harry Sharpe,1967-09-22,+44161 496 0148,jenna56@example.org,"Studio 5 Olivia motorway, Geraldineberg, East Midlands, NG8 6JP",ACC000000005,BR005,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:04:06.626053Z,83eb6b4e3627231f52b6a63731701075ee2f88e4e7b7469390018795f1eb3397
C00000006,Keith Francis Wilson,1984-05-13,(0141) 4960037,ndavey@example.org,"0 Welch spurs, Stewarttown, London, W12 6VP",ACC000000006,BR006,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:04:06.626053Z,6398faee4aa7bde53fa35426da09e04fe12e18ffc00938c2451eb59808025352
C00000007,Mitchell Janice Blackburn,1974-06-16,(0113)4960059,darrendouglas@example.com,"Studio 9 Simmons rapids, Port Ashley, Scotland, DD13 2GW",ACC000000007,BR007,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:04:06.626053Z,477f3d18475500e1265bfbe32f7d07f871d701f173f0a7d1ae0b487e4cab808f
C00000008,Raymond Marshall,1988-06-26,+4429 2018 0580,dayraymond@example.net,"714 Yvonne manor, East Justin, North West, SK2 7HW",ACC000000008,BR008,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:04:06.626053Z,a38a3b6dfdb02bf6e28c3abee2fa4874446dd37f92f5da189996918a4785930c
C00000009,Leanne Robert Stevens,1982-04-11,+44(0)9098790874,robertscarol@example.com,"Studio 88J Sara wells, East Jade, London, WC16 8FG",ACC000000009,BR009,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:04:06.626053Z,24ffe410a1199b6ebb79cd1dbfdbb039f269f4e0f124ec66dfb2a88e80d7c156
C00000010,Dean Reynolds,1978-02-07,+44(0)1914960387,damianrowe@example.net,"Flat 75H Ali union, Georgiachester, North West, CH20 7ZU",ACC000000011,BR011,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:04:06.626053Z,e447f401f075ceefda2936079e1069b97148957efc26bb6f6c94a093d19f4599


####Write Core to Unity Catalog Bronze


This writes/registers: bfsi_mdm_dev.bronze.core as a Unity Catalog Delta table.

In [0]:
(
    core_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.bronze.core"
    )
)

####Verify the Core Bronze table

Verify that Unity Catalog registered the Bronze table.

In [0]:
%sql
SHOW TABLES IN bfsi_mdm_dev.bronze

database,tableName,isTemporary
bronze,core,false


In [0]:
%sql


In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM bfsi_mdm_dev.bronze.core;

row_count
964240


In [0]:
%sql
SELECT
    customer_id,
    full_name,
    dob,
    batch_id,
    ingestion_timestamp,
    record_hash
FROM bfsi_mdm_dev.bronze.core
LIMIT 10;

customer_id,full_name,dob,batch_id,ingestion_timestamp,record_hash
C00000001,Connor Smith,1971-10-19,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:11:44.257778Z,8102a00df6062ec45f69cc250d4a9dc206186a10f2f9f93dbdf8faa0c30885b6
C00000002,Kevin Ruth Hayes,1970-05-26,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:11:44.257778Z,153bf97e864a23a140285f0ca4b002c28e0b95cf524a46206c55abe328a39ad6
C00000003,Jemma Dean,1957-03-03,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:11:44.257778Z,1f086542eeb84706310d64d94eaace4972020089c9e2fa7d2f0374363aa6273f
C00000004,Richard Duncan Chambers,1983-10-21,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:11:44.257778Z,45daba61a36460eae163d3f55785c3c1185ab30008440d965f7bbe90b8c8e8f5
C00000005,Harry Sharpe,1967-09-22,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:11:44.257778Z,83eb6b4e3627231f52b6a63731701075ee2f88e4e7b7469390018795f1eb3397
C00000006,Keith Francis Wilson,1984-05-13,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:11:44.257778Z,6398faee4aa7bde53fa35426da09e04fe12e18ffc00938c2451eb59808025352
C00000007,Mitchell Janice Blackburn,1974-06-16,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:11:44.257778Z,477f3d18475500e1265bfbe32f7d07f871d701f173f0a7d1ae0b487e4cab808f
C00000008,Raymond Marshall,1988-06-26,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:11:44.257778Z,a38a3b6dfdb02bf6e28c3abee2fa4874446dd37f92f5da189996918a4785930c
C00000009,Leanne Robert Stevens,1982-04-11,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:11:44.257778Z,24ffe410a1199b6ebb79cd1dbfdbb039f269f4e0f124ec66dfb2a88e80d7c156
C00000010,Dean Reynolds,1978-02-07,316d99a2-de25-4de9-b9de-a7bc440c6eba,2026-09-21T22:11:44.257778Z,e447f401f075ceefda2936079e1069b97148957efc26bb6f6c94a093d19f4599


####Verify Delta

In [0]:
%sql
DESCRIBE DETAIL bfsi_mdm_dev.bronze.core;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,df4a71af-56ca-46b8-b445-37519f072a90,bfsi_mdm_dev.bronze.core,null,abfss://bronze@stbfsimdmdev.dfs.core.windows.net/__unitystorage/schemas/564ea77d-f18a-40cc-ab50-11b4a4ef2701/tables/be86c92b-cfd8-440d-9a68-24d664a07197,2026-09-21T22:11:43.612Z,2026-09-21T22:11:56Z,List(),List(),1,73872259,"Map(io.unitycatalog.tableId -> be86c92b-cfd8-440d-9a68-24d664a07197, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


####FULL BRONZE LOAD

One batch_id identifies this complete execution.

Each source becomes one Unity Catalog Bronze Delta table.

In [0]:
batch_id = str(uuid.uuid4())

print(f"Starting Bronze batch: {batch_id}")

for system, (path, schema) in SOURCES.items():

    print("=" * 70)
    print(f"Processing source: {system.upper()}")
    print("=" * 70)

    bronze_df = read_bronze(
        relative_path=path,
        schema=schema,
        batch_id=batch_id,
    )

    # Count records for an explicit ingestion audit.
    row_count = bronze_df.count()

    print(
        f"{system.upper()} Bronze rows: "
        f"{row_count:,}"
    )

    (
        bronze_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"{CATALOG}.bronze.{system}"
        )
    )

    print(
        f"Written: "
        f"{CATALOG}.bronze.{system}"
    )

print("=" * 70)
print("BRONZE LOAD COMPLETE")
print(f"batch_id = {batch_id}")
print("=" * 70)

Starting Bronze batch: fb3a069b-715f-4323-8e39-5b86aaf38d4f
Processing source: CORE
CORE Bronze rows: 964,240
Written: bfsi_mdm_dev.bronze.core
Processing source: CRM
CRM Bronze rows: 660,223
Written: bfsi_mdm_dev.bronze.crm
Processing source: KYC
KYC Bronze rows: 811,892
Written: bfsi_mdm_dev.bronze.kyc
Processing source: WEALTH
WEALTH Bronze rows: 202,544
Written: bfsi_mdm_dev.bronze.wealth
BRONZE LOAD COMPLETE
batch_id = fb3a069b-715f-4323-8e39-5b86aaf38d4f


####Verify all four tables

In [0]:
%sql
SELECT 'CORE' AS source_system,
       COUNT(*) AS row_count
FROM bfsi_mdm_dev.bronze.core

UNION ALL

SELECT 'CRM',
       COUNT(*)
FROM bfsi_mdm_dev.bronze.crm

UNION ALL

SELECT 'KYC',
       COUNT(*)
FROM bfsi_mdm_dev.bronze.kyc

UNION ALL

SELECT 'WEALTH',
       COUNT(*)
FROM bfsi_mdm_dev.bronze.wealth;

source_system,row_count
CORE,964240
CRM,660223
KYC,811892
WEALTH,202544


In [0]:
%sql
SELECT
    (
        (SELECT COUNT(*) FROM bfsi_mdm_dev.bronze.core)
      + (SELECT COUNT(*) FROM bfsi_mdm_dev.bronze.crm)
      + (SELECT COUNT(*) FROM bfsi_mdm_dev.bronze.kyc)
      + (SELECT COUNT(*) FROM bfsi_mdm_dev.bronze.wealth)
    ) AS total_bronze_rows;

total_bronze_rows
2638899


####Verify the metadata columns

In [0]:
%sql
SELECT
    *
FROM (
    SELECT
        'core' AS source_table,
        batch_id,
        ingestion_timestamp
    FROM bfsi_mdm_dev.bronze.core
    LIMIT 1
) core_sample

UNION ALL

SELECT
    *
FROM (
    SELECT
        'crm' AS source_table,
        batch_id,
        ingestion_timestamp
    FROM bfsi_mdm_dev.bronze.crm
    LIMIT 1
) crm_sample

UNION ALL

SELECT
    *
FROM (
    SELECT
        'kyc' AS source_table,
        batch_id,
        ingestion_timestamp
    FROM bfsi_mdm_dev.bronze.kyc
    LIMIT 1
) kyc_sample

UNION ALL

SELECT
    *
FROM (
    SELECT
        'wealth' AS source_table,
        batch_id,
        ingestion_timestamp
    FROM bfsi_mdm_dev.bronze.wealth
    LIMIT 1
) wealth_sample;

source_table,batch_id,ingestion_timestamp
core,fb3a069b-715f-4323-8e39-5b86aaf38d4f,2026-09-21T22:40:42.277579Z
crm,fb3a069b-715f-4323-8e39-5b86aaf38d4f,2026-09-21T22:40:53.440494Z
kyc,fb3a069b-715f-4323-8e39-5b86aaf38d4f,2026-09-21T22:41:02.42712Z
wealth,fb3a069b-715f-4323-8e39-5b86aaf38d4f,2026-09-21T22:41:11.728408Z


####Verify the Bronze tables are Delta tables.

In [0]:
%sql
DESCRIBE DETAIL bfsi_mdm_dev.bronze.core;

DESCRIBE DETAIL bfsi_mdm_dev.bronze.crm;

DESCRIBE DETAIL bfsi_mdm_dev.bronze.kyc;

DESCRIBE DETAIL bfsi_mdm_dev.bronze.wealth;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,cb0a652c-c198-4dfa-a740-b1db009001cc,bfsi_mdm_dev.bronze.wealth,null,abfss://bronze@stbfsimdmdev.dfs.core.windows.net/__unitystorage/schemas/564ea77d-f18a-40cc-ab50-11b4a4ef2701/tables/2e5de037-2bd9-47cf-913b-0f81cd198635,2026-09-21T22:41:11.52Z,2026-09-21T22:41:15Z,List(),List(),1,15684148,"Map(io.unitycatalog.tableId -> 2e5de037-2bd9-47cf-913b-0f81cd198635, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false
